All data is loaded directly from JSON files in `mock_eng_data/` — no backend API calls.
The Zingplay vllm endpoint is used for `prompt_logprobs`.

| File | Topic | Case |
|------|-------|------|
| `real_outcome.json` | VNG taxi allowance (Vietnamese) | Baseline validation |
| `case1_full_answer_relevant_context.json` | Sports nutrition — protein | Full answer + relevant context |
| `case2_answer_irrelevant_context.json` | Python — string reversal | Answer correct, context irrelevant |
| `case3_relevant_context_answer_misses_query.json` | Sleep science — bedtime | Context relevant, answer misses query |
| `case4_partial_answer_partial_context.json` | Home gardening — tomatoes | Partial answer + partial context |

In [1]:
import sys, json
from pathlib import Path
from IPython.display import display

sys.path.insert(0, str(Path('..').resolve()))

from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')

# Let the module configure itself from .env (local-model-mini + Zingplay endpoint)
import context_attribution.context_cite as cc_module
from context_attribution.context_cite import prepare_inputs, attribute_response

print(f'LLM endpoint        : {cc_module.LLM_URL}')
print(f'Model               : {cc_module.MODEL}')
print(f'API key set         : {bool(cc_module.API_KEY)}')
print(f'Max response chars  : {cc_module.MINI_MAX_RESPONSE_CHARS}')
print(f'Max context chars   : {cc_module.MINI_MAX_CONTEXT_CHARS}')

DATA_DIR = Path('mock_eng_data')

LLM endpoint        : https://chat.zingplay.com/api/v1/chat/completions
Model               : local-model-mini
API key set         : True
Max response chars  : 120
Max context chars   : 900


[nltk_data] Downloading package punkt_tab to /home/coder/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
def inspect_case(data, query, label):
    '''Print case summary and return (context, response) ready for attribution.'''
    context, response = prepare_inputs(data)
    print('=' * 72)
    print(label)
    print('=' * 72)
    print()
    print('Query:')
    print(' ', query)
    print()
    print(f'Answer ({len(response)} chars):')
    print(response)
    print()
    n_src = len(data['knowledge_sources'])
    print(f'Knowledge sources ({n_src} chunks):')
    for i, src in enumerate(data['knowledge_sources']):
        doc     = src.get('document_id') or '<no doc>'
        score   = src.get('score', 0)
        content = src.get('content', '')
        snip    = ' '.join(content[:120].split()) if content else '<empty>'
        print(f'  [{i}] score={score:.3f}  {doc}')
        print(f'       {snip}...')
    return context, response

---
## Baseline: `real_outcome.json`

Data is from a chat response from backend API, answering a question from the test set. 

In [3]:
data  = json.load(open(DATA_DIR / 'real_outcome.json'))
query = 'T\u00f4i \u0111\u01b0\u1ee3c \u0111i\u1ec1u ch\u1ec9nh c\u1ea5p b\u1eadc ng\u00e0y 22/08. Tr\u1ee3 c\u1ea5p taxi m\u1edbi s\u1ebd c\u00f3 hi\u1ec7u l\u1ef1c t\u1eeb th\u00e1ng n\u00e0o?'

context, response = inspect_case(
    data, query,
    'BASELINE  |  real_outcome.json  |  VNG taxi allowance (Vietnamese)'
)

print()
print('Running context attribution (32 ablations)...')
result = attribute_response(context, query, response, num_ablations=32, verbose=True)
display(result)

Chunks: 3/5 | Context: 700 chars | Response: 120 chars
BASELINE  |  real_outcome.json  |  VNG taxi allowance (Vietnamese)

Query:
  Tôi được điều chỉnh cấp bậc ngày 22/08. Trợ cấp taxi mới sẽ có hiệu lực từ tháng nào?

Answer (120 chars):
Dựa trên chính sách trợ cấp taxi của VNG:

**Nhân viên sẽ bắt đầu được nhận trợ cấp taxi theo định mức cấp bậc mới từ th

Knowledge sources (5 chunks):
  [0] score=1.750  CS Phu╠üc lo╠¢╠úi/CS Phu╠üc lo╠¢╠úi ca╠ån ba╠ën (chung).md
       ### 4.2. Trợ cấp taxi Tùy theo cấp bậc hoặc yêu cầu công việc đang đảm nhận, Công ty hỗ trợ Trợ cấp taxi cho Nhân viên...
  [1] score=0.898  <no doc>
       <empty>...
  [2] score=0.839  CS Phu╠üc lo╠¢╠úi/CS Phu╠üc lo╠¢╠úi ca╠ån ba╠ën (chung).md
       ### 4.3. Trợ cấp xe ôtô riêng Đối với các Nhân viên thuộc cấp bậc từ 4.3 trở lên, Công ty cung cấp phúc lợi xe ôtô riên...
  [3] score=0.824  CS Phu╠üc lo╠¢╠úi/CS Phu╠üc lo╠¢╠úi ca╠ån ba╠ën (chung).md
       ### 4.1. Trợ cấp điện thoại Tùy theo cấp bậc 

Attributed: Dựa trên chính sách trợ cấp taxi của VNG:

**Nhân viên sẽ bắt đầu được nhận trợ cấp taxi theo định mức cấp bậc mới từ th


  0%|          | 0/32 [00:00<?, ?it/s]

/home/coder/dairy_vng/.venv/lib/python3.11/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
/home/coder/dairy_vng/.venv/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


,Score,Source
0,14.307,"Đối với các Nhân viên thuộc cấp bậc từ 4.3 trở lên, Công ty cung cấp phúc lợi xe ôtô riêng theo định mức, theo nhu cầu đi lại phục vụ công việc."
1,11.612,Trợ cấp taxi
2,4.948,Chi tiết liên quan được qui định trong Chính sách hỗ trợ taxi do phòng AF ban hành.
3,0.000,Chi tiết liên quan được qui định trong Chính sách xe ô tô riêng.
4,0.000,Trợ cấp xe ôtô riêng
5,-0.000,### 4.3.
6,-1.510,Nhân viên được cung cấp thẻ taxi hoặc tài khoản sử dụng app taxi đã liên kết sẵn tài khoản thanh toán của Công ty.
7,-9.344,"Tùy theo cấp bậc hoặc yêu cầu công việc đang đảm nhận, Công ty hỗ trợ Trợ cấp taxi cho Nhân viên kể từ tháng gia nhập Công ty."
8,-12.748,### 4.2.
9,-29.463,"Trường hợp Nhân viên được thăng cấp, trợ cấp taxi sẽ hiệu lực từ tháng kế tiếp tháng hiệu lực thăng cấp."


---
## Case 1: Full Answer + Relevant Context
**Topic:** Sports nutrition — daily protein intake for muscle building

The retrieved chunks directly contain the answer: 1.6–2.2 g/kg/day, meal timing,
protein sources. The response is complete and drawn from those chunks.

In [4]:
data  = json.load(open(DATA_DIR / 'case1_full_answer_relevant_context.json'))
query = 'How much protein should I eat per day to build muscle?'

context, response = inspect_case(
    data, query,
    'CASE 1  |  Full answer + relevant context  |  Sports Nutrition'
)

print()
print('Running context attribution (32 ablations)...')
result = attribute_response(context, query, response, num_ablations=32, verbose=True)
display(result)

Chunks: 5/5 | Context: 875 chars | Response: 103 chars
CASE 1  |  Full answer + relevant context  |  Sports Nutrition

Query:
  How much protein should I eat per day to build muscle?

Answer (103 chars):
Aim for 1.6–2.2 g of protein per kg of bodyweight each day. Spread it across 3–5 meals of 20–40 g each.

Knowledge sources (5 chunks):
  [0] score=1.923  Sports Nutrition/protein_recommendations.md
       ### Daily Protein Target Research supports 1.6–2.2 g of protein per kg of bodyweight per day for muscle hypertrophy. In...
  [1] score=1.521  Sports Nutrition/protein_recommendations.md
       ### Meal Frequency and Dose Spreading protein across 3–5 meals of 20–40 g each produces more muscle protein synthesis t...
  [2] score=1.205  Sports Nutrition/protein_recommendations.md
       ### Post-Workout Timing Eating 20–40 g of protein within two hours of training significantly boosts muscle protein synt...
  [3] score=0.889  Sports Nutrition/protein_sources.md
       ### Protein Sources 

Attributed: Aim for 1.6–2.2 g of protein per kg of bodyweight each day. Spread it across 3–5 meals of 20–40 g each.


  0%|          | 0/32 [00:00<?, ?it/s]

/home/coder/dairy_vng/.venv/lib/python3.11/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
/home/coder/dairy_vng/.venv/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


,Score,Source
0,10.636,Spreading protein across 3–5 meals of 20–40 g each produces more muscle protein synthesis than the same total consumed in fewer large servings.
1,5.341,Research supports 1.6–2.2 g of protein per kg of bodyweight per day for muscle hypertrophy.
2,1.015,### Post-Workout Timing
3,0.876,### Daily Protein Target
4,0.393,Leucine-rich sources such as eggs and whey are ideal.
5,0.386,### Protein Sources
6,-0.000,Higher protein intake also improves dietary adherence.
7,-0.000,"Chicken breast, eggs, Greek yogurt, and tuna are high-quality sources rich in all essential amino acids."
8,0.000,### Protein While Cutting
9,0.000,Well-planned plant diets can meet the same targets.


---
## Case 2: Answer Correct, Context Irrelevant
**Topic:** Python string reversal; retrieved content is about coffee




In [5]:
data  = json.load(open(DATA_DIR / 'case2_answer_irrelevant_context.json'))
query = 'How do I reverse a string in Python?'

context, response = inspect_case(
    data, query,
    'CASE 2  |  Answer correct, context irrelevant  |  Python Programming'
)

print()
print('Running context attribution (32 ablations)...')
result = attribute_response(context, query, response, num_ablations=32, verbose=True)
display(result)

Chunks: 5/5 | Context: 855 chars | Response: 111 chars
CASE 2  |  Answer correct, context irrelevant  |  Python Programming

Query:
  How do I reverse a string in Python?

Answer (111 chars):
The easiest way to reverse a string in Python is slicing: text[::-1]. This returns the string in reverse order.

Knowledge sources (5 chunks):
  [0] score=0.712  Coffee Guide/bean_selection.md
       ### Choosing Coffee Beans Arabica beans produce sweeter, more complex flavour than Robusta. Light roasts are acidic and...
  [1] score=0.589  Coffee Guide/grind_extraction.md
       ### Grind Size and Extraction Finer grinds increase surface area for faster extraction and suit espresso. Coarser grind...
  [2] score=0.444  Coffee Guide/brew_temperature.md
       ### Water Temperature Ideal brewing temperature is 90–96 °C. Water below 85 °C yields sour, under-extracted coffee; abo...
  [3] score=0.391  Coffee Guide/brew_methods.md
       ### Brew Time Espresso extracts in 25–30 seconds. French press ste

Attributed: The easiest way to reverse a string in Python is slicing: text[::-1]. This returns the string in reverse order.


  0%|          | 0/32 [00:00<?, ?it/s]

/home/coder/dairy_vng/.venv/lib/python3.11/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
/home/coder/dairy_vng/.venv/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


,Score,Source
0,0.266,A standard ratio is 1:15 to 1:17 by weight.
1,0.156,Adjusting this ratio is the most direct way to control brew strength without changing grind or temperature.
2,0.140,Ideal brewing temperature is 90–96 °C.
3,0.116,Finer grinds increase surface area for faster extraction and suit espresso.
4,0.005,Longer steeping generally increases body and bitterness.
5,0.000,### Brew Time
6,0.000,Espresso extracts in 25–30 seconds.
7,0.000,Pour-over takes 3–4 minutes.
8,0.000,French press steeps for 4 minutes.
9,0.000,### Grind Size and Extraction


---
## Case 3: Relevant Context, Answer Misses the Query
**Topic:** Sleep science — user asked *what time* to go to sleep; context directly
answers this (10–11:30 PM, 90-min cycle math); but the response instead discusses
*why* sleep matters and generic tips without giving a time.

In [6]:
data  = json.load(open(DATA_DIR / 'case3_relevant_context_answer_misses_query.json'))
query = 'What time should I go to sleep to wake up feeling rested?'

context, response = inspect_case(
    data, query,
    'CASE 3  |  Relevant context, answer misses query  |  Sleep Science'
)

print()
print('Running context attribution (32 ablations)...')
result = attribute_response(context, query, response, num_ablations=32, verbose=True)
display(result)

Chunks: 5/5 | Context: 898 chars | Response: 111 chars
CASE 3  |  Relevant context, answer misses query  |  Sleep Science

Query:
  What time should I go to sleep to wake up feeling rested?

Answer (111 chars):
Sleep supports memory, immune function, and emotional wellbeing. Most adults need 7–9 hours of sleep per night.

Knowledge sources (5 chunks):
  [0] score=1.835  Sleep Science/circadian_rhythms.md
       ### Optimal Bedtime For most adults, the ideal sleep window is 10:00–11:30 PM, when melatonin secretion peaks naturally...
  [1] score=1.613  Sleep Science/sleep_cycles.md
       ### Sleep Cycle Maths One sleep cycle is 90 minutes. To wake at 6:30 AM refreshed, go to sleep at 9:00 PM, 10:30 PM, or...
  [2] score=1.400  Sleep Science/sleep_duration.md
       ### Recommended Duration Adults aged 18–60 need at least 7 hours nightly; adults 61–64 should target 7–9 hours. Regular...
  [3] score=1.189  Sleep Science/chronotypes.md
       ### Chronotypes Early chronotypes prefer to sle

Attributed: Sleep supports memory, immune function, and emotional wellbeing. Most adults need 7–9 hours of sleep per night.


  0%|          | 0/32 [00:00<?, ?it/s]

/home/coder/dairy_vng/.venv/lib/python3.11/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
/home/coder/dairy_vng/.venv/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


,Score,Source
0,0.230,A cool bedroom of 16–19 °C also improves sleep quality.
1,0.170,"To wake at 6:30 AM refreshed, go to sleep at 9:00 PM, 10:30 PM, or midnight to complete a whole number of cycles."
2,0.063,Adults aged 18–60 need at least 7 hours nightly; adults 61–64 should target 7–9 hours.
3,0.062,Early chronotypes prefer to sleep and wake earlier; night owls prefer later schedules.
4,0.000,Chronotype is partly genetic and tends to shift later during the teenage years.
5,-0.000,### Chronotypes
6,0.000,### Evening Routine
7,0.000,### Sleep Cycle Maths
8,0.000,Regularly sleeping fewer than 7 hours is associated with health risks.
9,-0.000,### Recommended Duration


---
## Case 4: Partial Answer + Partial Context
**Topic:** Home gardening — user asked about *growing tomatoes* AND *what pests to
watch out for*. Context covers growing well (soil, watering, fertilizing) but has
no dedicated pest information. The answer covers growing fully but only vaguely
mentions pests.

In [7]:
data  = json.load(open(DATA_DIR / 'case4_partial_answer_partial_context.json'))
query = 'How do I grow tomatoes and what pests should I watch out for?'

context, response = inspect_case(
    data, query,
    'CASE 4  |  Partial answer + partial context  |  Home Gardening (tomatoes)'
)

print()
print('Running context attribution (32 ablations)...')
result = attribute_response(context, query, response, num_ablations=32, verbose=True)
display(result)

Chunks: 5/5 | Context: 873 chars | Response: 116 chars
CASE 4  |  Partial answer + partial context  |  Home Gardening (tomatoes)

Query:
  How do I grow tomatoes and what pests should I watch out for?

Answer (116 chars):
Plant in full sun with well-drained soil at pH 6.0–6.8. Water 2–3 times weekly and watch for insects or leaf damage.

Knowledge sources (5 chunks):
  [0] score=1.788  Home Gardening/growing_tomatoes.md
       ### Sunlight and Soil Tomatoes need at least 6–8 hours of direct sun per day. Plant in well-drained soil with a pH of 6...
  [1] score=1.543  Home Gardening/growing_tomatoes.md
       ### Watering and Support Water deeply 2–3 times per week rather than lightly every day to encourage deep root growth. S...
  [2] score=0.935  Home Gardening/growing_tomatoes.md
       ### Fertilizing Tomatoes are heavy feeders. Add compost before planting and switch to a low-nitrogen fertilizer once fl...
  [3] score=0.621  Home Gardening/soil_preparation.md
       ### Raised Beds R

Attributed: Plant in full sun with well-drained soil at pH 6.0–6.8. Water 2–3 times weekly and watch for insects or leaf damage.


  0%|          | 0/32 [00:00<?, ?it/s]

/home/coder/dairy_vng/.venv/lib/python3.11/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
/home/coder/dairy_vng/.venv/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


,Score,Source
0,8.512,Plant in well-drained soil with a pH of 6.0–6.8 enriched with compost before planting.
1,5.306,Tomatoes need at least 6–8 hours of direct sun per day.
2,0.000,### Raised Beds
3,-0.000,"Raised beds with a mix of topsoil, compost, and perlite offer excellent drainage and warm up faster in spring."
4,-0.000,Stake or cage plants once they reach 30 cm tall.
5,0.000,### Fertilizing
6,0.000,Water deeply 2–3 times per week rather than lightly every day to encourage deep root growth.
7,0.000,### Watering and Support
8,-0.230,### Companion Planting
9,-0.244,Add compost before planting and switch to a low-nitrogen fertilizer once flowering begins to favour fruit over leaf growth.
